In [ ]:
######   preprocessing
import os
import pandas as pd

# ----------- PATHS -------------
input_folder = "../../data/raw/processed_data"
output_folder = "../../data/raw/lstm_data"

os.makedirs(output_folder, exist_ok=True)
# --------------------------------

for file in os.listdir(input_folder):
    if file.endswith(".csv"):
        input_path = os.path.join(input_folder, file)

        # Load csv
        df = pd.read_csv(input_path)

        # Fill only Unit1 & Unit2 with -1
        if "Unit1" in df.columns:
            df["Unit1"] = df["Unit1"].fillna(-1)

        if "Unit2" in df.columns:
            df["Unit2"] = df["Unit2"].fillna(-1)

        # Save cleaned version to output directory
        output_path = os.path.join(output_folder, file)
        df.to_csv(output_path, index=False)

        print(f"Cleaned and saved: {file}")

print("✅ ALL FILES PROCESSED & SAVED IN data/lstm/")


In [ ]:
####### find max sequences length
import os
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler

data_dir = "../../data/raw/lstm_data/"

sequences = []
seq_lengths = []

for file in os.listdir(data_dir):
    if file.endswith(".csv"):
        df = pd.read_csv(os.path.join(data_dir, file))

        # Convert to numpy array
        seq = df.values.astype(float)

        sequences.append(seq)
        seq_lengths.append(len(seq))

MAX_LEN = max(seq_lengths)
print("Max sequence length =", MAX_LEN)


In [ ]:
######## padding
PAD_VALUE = -999

padded_sequences = []
for seq in sequences:
    pad_len = MAX_LEN - len(seq)
    if pad_len > 0:
        pad = np.full((pad_len, seq.shape[1]), PAD_VALUE)
        seq = np.vstack([seq, pad])
    padded_sequences.append(seq)

padded_sequences = np.array(padded_sequences)
print("Shape after padding:", padded_sequences.shape)


In [ ]:
########## scaling
IGNORE_COLS = [0, 1]    # unit1, unit2
scaler = StandardScaler()

# 1. Reshape to 2D for scaling
N, T, F = padded_sequences.shape
flat = padded_sequences.reshape(-1, F)

# 2. Identify normal (non-pad) rows
valid_mask = (flat[:, 0] != PAD_VALUE)

# 3. Select only valid rows, and only non-ignore columns
cols_to_scale = [i for i in range(F) if i not in IGNORE_COLS]

valid_data = flat[valid_mask][:, cols_to_scale]

# 4. Fit scaler
scaler.fit(valid_data)

# 5. Transform ONLY valid rows, ONLY selected columns
flat_scaled = flat.copy()
flat_scaled[valid_mask][:, cols_to_scale] = scaler.transform(valid_data)

# 6. Restore padded rows unchanged
flat_scaled[~valid_mask] = PAD_VALUE

# 7. Reshape back to (patients, max_len, features)
scaled_sequences = flat_scaled.reshape(N, T, F)


In [ ]:
import sys
print(sys.version)
!pip install tensorflow==2.12

In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Masking

X = scaled_sequences
y = np.array(labels)   # you must supply patient-level labels

model = Sequential([
    Masking(mask_value=PAD_VALUE, input_shape=(MAX_LEN, F)),
    LSTM(64, return_sequences=False),
    Dense(1, activation='sigmoid')
])

model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])
model.summary()


In [ ]:
import sys
print(sys.executable)

In [ ]:
%pip uninstall -y numpy

In [ ]:
%pip install numpy==1.25.2

In [ ]:
%pip uninstall -y keras


In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential

print(np.__version__)
print(tf.__version__)


In [ ]:
!python -m pip install --upgrade pip


In [ ]:
%pip uninstall -y tensorflow keras


In [ ]:
%pip install tensorflow==2.12.0


In [ ]:
import tensorflow 
print(tf.__version__)

from tensorflow.keras.models import Sequential
print("Keras import successful")

In [ ]:
import sys
print(sys.executable)


In [ ]:
%pip list



In [ ]:
%pip install -r ../../requirements.txt
